In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
#Importing dataset
df = pd.read_csv("Steel_industry_data.csv")

In [4]:
df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


In [9]:
#Defining target and features
y=df["Usage_kWh"]
features = [
    "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh",
    "Lagging_Current_Power_Factor",
    "Leading_Current_Power_Factor",
    "NSM"
]
X = df[features]

In [10]:
#Train/test split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

In [14]:
#Cross validation strategy
from sklearn.model_selection import KFold, cross_val_score
cv = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

In [19]:
#importing necessary libraries
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import(
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import(
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [23]:
#Building a reusable evaluation function that calculates, for a given model, the cross-validation performance, the fitting performance, the test R2, MAE and RMSE
def evaluate_model(name, model):
    
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring = "r2"
    )

    #fit model on complete training set
    model.fit(X_train, y_train)

    #training predictions
    train_pred = model.predict(X_train)

    #test predictions
    test_pred = model.predict(X_test)

    #metrics
    train_r2 = r2_score(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(
        mean_squared_error(y_test,test_pred)
    )
    test_r2 = r2_score(y_test,test_pred)

    return{
        "Model": name,
        "Train R2": train_r2,
        "CV R2": cv_scores.mean(),
        "CV std": cv_scores.std(),
        "Test MAE": test_rmse,
        "Test R2": test_r2
    }

In [24]:
#Linear model
linear_model = LinearRegression()
linear_results = evaluate_model(
    "Linear regression",
    linear_model
)

In [25]:
#Ridge model
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])
ridge_results = evaluate_model(
    "Ridge",
    ridge_model
)

In [26]:
#Random forest model
rf_model = RandomForestRegressor(
    n_estimators = 300,
    random_state = 42,
    n_jobs = -1
)
rf_results = evaluate_model(
    "Random forest",
    rf_model
)

In [27]:
#Gradient boosting model
gb_model = GradientBoostingRegressor(
    random_state = 42
)
gb_results = evaluate_model(
    "Gradient boosting",
    gb_model
)

In [31]:
#XGBoost model
from xgboost import XGBRegressor
xgb_model = XGBRegressor(
    n_estimators = 300,
    learning_rate = 0.05,
    max_depth = 3,
    subsample = 0.8,
    colsample_bytree = 0.8,
    random_state = 42,
    objective = "reg:squarederror"
)
xgb_results = evaluate_model(
    "XGBoost",
    xgb_model
)

In [32]:
#baseline model - simple average
from sklearn.dummy import DummyRegressor
baseline_model = DummyRegressor(
    strategy = "mean"
)

baseline_results = evaluate_model(
    "Mean baseline",
    baseline_model
)

In [34]:
#combining the results from different models
results = pd.DataFrame ([
    baseline_results,
    linear_results,
    ridge_results,
    rf_results,
    gb_results,
    xgb_results
])

In [39]:
#Evaluating generalization gap
results["Generalization Gap"] = (
    results["Train R2"] - results["CV R2"]
)
results.sort_values(
    "CV R2",
    ascending = False
)

,Model,Train R2,CV R2,CV std,Test MAE,Test R2,Generalization Gap
3,Random forest,0.999949,0.999485,0.000056,0.624992,0.999656,0.000464
5,XGBoost,0.996369,0.995958,0.000117,2.085049,0.996175,0.000411
4,Gradient boosting,0.996209,0.995956,0.000275,2.115303,0.996064,0.000252
2,Ridge,0.912876,0.912803,0.002896,10.113885,0.910013,0.000073
1,Linear regression,0.912876,0.912803,0.002896,10.113856,0.910013,0.000073
0,Mean baseline,0.000000,-0.000086,0.000038,33.719081,-0.000224,0.000086


##Conclusions
Random forest is clearly the winner here, with a CV R2 of 0.999485. It means its predictions on unseen CV folds explain essentially all the variation in Usage_kWh. Additionally, the Generalization gap is extremely small as well, and the CV std is tiny, meaning that the model is stable across the five folds. 
Although we should not use the test performance to choose the model, the test result is consistent with the CV result.


In [43]:
##Hyperparameter tuning example - Ridge model
from sklearn.model_selection import GridSearchCV

#define the pipeline
ridge_pipeline = Pipeline([
    ("scaler",StandardScaler()),
    ("ridge", Ridge())
])

#define the grid
param_grid = {
    "ridge__alpha": [
        0.01,
        0.1,
        1,
        10,
        100
    ] 
}

#search. it selects the smallest MAE based on what we gave it
ridge_search = GridSearchCV(
    ridge_pipeline,
    param_grid = param_grid,
    cv=cv,
    scoring = "neg_mean_absolute_error"
)

ridge_search.fit(
    X_train,
    y_train
)

print("Best parameters:", ridge_search.best_params_)
print("Best CV MAE:", -ridge_search.best_score_)

ridge_cv_results = pd.DataFrame(
    ridge_search.cv_results_
)

ridge_cv_results["CV_MAE"] = (
    -ridge_cv_results["mean_test_score"]
)

ridge_cv_results[
    [
        "param_ridge__alpha",
        "CV_MAE",
        "rank_test_score"
    ]
]

Best parameters: {'ridge__alpha': 100}
Best CV MAE: 6.769760056477739


,param_ridge__alpha,CV_MAE,rank_test_score
0,0.01,6.784437,5
1,0.10,6.784417,4
2,1.00,6.784215,3
3,10.00,6.782276,2
4,100.00,6.769760,1


In [46]:
#random forest hyperparameter optimization
#the matrix of cominbations is huge, since we have 3 estimators x 4 depths x 3 min leafs x 3 features (108 combinations).
#each combination is then fit for 5 times, since we have 5 folds, so 540 model fits
#for models like random forests, better to run a randomized search 

from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.5, 1.0]
}

rf = RandomForestRegressor (
    random_state = 42,
    n_jobs = -1
)

rf_search = RandomizedSearchCV(
    estimator = rf,
    param_distributions = param_grid,
    n_iter = 30,
    cv = cv,
    scoring = "neg_mean_absolute_error",
    n_jobs = -1
)

rf_search.fit(
    X_train,
    y_train
)

print("Best parameters:")
print(rf_search.best_params_)

print("Best CV MAE:")
print(-rf_search.best_score_)

#with n_iter you try 30 randomly selected hyperparameter combinations. 

Best parameters:
{'n_estimators': 500, 'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': None}
Best CV MAE:
0.22828995858051981


In [47]:
#now we compare the default Random Forest and optimized random forest using the same metric.
default_rf_cv_mae = -cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
).mean()

print("Default RF CV MAE:", default_rf_cv_mae)
print("Optimized RF CV MAE:", -rf_search.best_score_)

Default RF CV MAE: 0.2290972769508613
Optimized RF CV MAE: 0.22828995858051981


In [48]:
best_rf = rf_search.best_estimator_

In [49]:
best_rf_cv_r2 = cross_val_score(
    best_rf,
    X_train,
    y_train,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

print("Optimized RF CV R²:", best_rf_cv_r2.mean())
print("Optimized RF CV std:", best_rf_cv_r2.std())

Optimized RF CV R²: 0.999484474347329
Optimized RF CV std: 5.727864265412586e-05


#the hyperparameter optimization for the RF model essentially did not improve much the standard model. The default model was already performing near the optimum. 